# Phase 6 (Abstractive Summarisation) — c-1 decoder-only + AdaLoRA

Input: `[document part] [SEP] [summary] </s>`, loss on the summary tokens and `</s>` only, 2048 positions.

Run the cells in order on Kaggle (GPU + internet on, `HF` secret set, Phase 1 c-1 output attached; check `ROOT` in Cell A).
**Fill in only section [H] of Cell E1** (every `None  # <<< FILL`): `SEP`, `WINDOW`, `STRIDE`, `MAX_WINDOWS`, and the AdaLoRA values
`ADA_INIT_RANK`, `ADA_TARGET_RANK`, `TINIT_FRAC`, `TFINAL_FRAC`, `DELTA_T`, `BETA1`, `BETA2`, `ORTH_REG`.
Training and decoding values are copied from your c-6 LORMA run. The cell stops and lists anything still unset.


In [1]:
# --- Phase 1 c-1 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
# --- Phase 1 c-1 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


In [3]:
# --- Phase 1 c-1 notebook, cell 21 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


In [4]:
# =====================================================================================
# CELL A — load the Phase 1 c-1 decoder and its tokenizer
# Expect <All keys matched successfully>, 110.98M params, vocab 10000, d_model 1024, 8 layers, max_seq_len 2048
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

# [c-1] checkpoint folder is legal-gpt2-final (c-6 was model_encdec); class is DecoderLM + LMConfig
tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| d_ff", cfg.d_ff, "| max_seq_len", cfg.max_seq_len,
      "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)


<All keys matched successfully>
params 110.979072 M | vocab 10000 | d_model 1024 | layers 8 | d_ff 4096 | max_seq_len 2048 | norm_first False | dropout 0.1


In [5]:
# =====================================================================================
# CELL B — per-token vectors from the c-1 decoder + sanity check
# [c-1] c-1 has no encoder: the features are the decoder's last hidden states (after ln_f),
#       with the LM head skipped. Each token only sees the tokens to its LEFT (causal).
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

# Sanity check: the LM head on these features must give the full model's logits
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)


max diff 0.0 (1, 64, 1024)


In [6]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 6 (Summarisation) data: train / held-out val / test
# One record = one long judgment ("document") -> one reference summary ("summary").
# There is NO dev split in Phase 6, so a fixed, seeded part of train is held out as val.
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, random
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P6_DIR = "Phase_6_Summ"
print("Phase 6 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_6" in f])

VAL_FRAC   = 0.10   # share of train held out for validation (no official dev split)
SPLIT_SEED = 42     # fixed, so every run / backbone uses the SAME held-out documents

def get6(name):
    """Reads Phase_6_Summ/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P6_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

TEXT_FIELDS    = ("document", "text", "judgment", "case_text", "source")
SUMMARY_FIELDS = ("summary", "abstract", "target", "headnote")

def as_docs(raw, name):
    """-> {doc_id: {"text": str, "summary": str or None}}, in file order.
    Accepts a list of dicts, {doc_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "document": [...], ("summary": [...])}.
    Repeated ids: exact duplicates dropped; differing repeats kept as "<id>#dup<n>" (train),
    dropped with a warning on test (exactly one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    docs, n_exact, n_diff = {}, 0, 0
    for k, d in items:
        if isinstance(d, str):                       # {id: "document text"} layout
            d = {"document": d}
        assert isinstance(d, dict), f"doc {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"doc {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):
            text = " ".join(map(str, text))
        sf = next((f for f in SUMMARY_FIELDS if f in d), None)
        summ = d[sf] if sf else None
        if isinstance(summ, list):
            summ = " ".join(map(str, summ))
        rec = {"text": str(text), "summary": None if summ is None else str(summ)}
        if k in docs:
            if docs[k] == rec:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            j = 1
            while f"{k}#dup{j}" in docs:
                j += 1
            k = f"{k}#dup{j}"
        docs[k] = rec
    print(f"{name}: {len(items)} records -> {len(docs)} docs | exact duplicates dropped {n_exact} | "
          f"repeated ids with different content kept {n_diff}")
    return docs

_raw = get6("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)
full_train = as_docs(_raw, "train")
test = as_docs(get6("Test/test.json"), "test")
del _raw

_empty = [k for k, d in full_train.items() if not (d["summary"] or "").strip()]
if _empty:
    print(f"dropping {len(_empty)} train docs with an empty summary")
    for k in _empty:
        del full_train[k]
_tl = sum(d["summary"] is not None for d in test.values())
if _tl:
    print(f"test has a summary field on {_tl} docs (ignored, set to None)")
for d in test.values():
    d["summary"] = None

# Held-out validation split (seeded shuffle of train ids)
_ids = list(full_train)
random.Random(SPLIT_SEED).shuffle(_ids)
_n_val = max(1, int(round(VAL_FRAC * len(_ids))))
VAL_IDS = set(_ids[:_n_val])
train = {k: d for k, d in full_train.items() if k not in VAL_IDS}
val   = {k: d for k, d in full_train.items() if k in VAL_IDS}
print(f"\ntrain {len(train)} | val (held out) {len(val)} | test {len(test)}")

for name, split in (("train", train), ("val", val), ("test", test)):
    w = np.array([len(d["text"].split()) for d in split.values()])
    line = f"{name:5s} document words: median {np.median(w):.0f} p90 {np.percentile(w, 90):.0f} max {w.max()}"
    if split is not test:
        s = np.array([len(d["summary"].split()) for d in split.values()])
        r = s / np.maximum(1, w)
        line += (f" | summary words: min {s.min()} median {np.median(s):.0f} p90 {np.percentile(s, 90):.0f} "
                 f"max {s.max()} | summary/document ratio median {np.median(r):.3f}")
    print(line)
print("first test ids:", list(test)[:3])
print(f"train∩test ids {len(set(full_train) & set(test))}")


Phase 6 files on HF: ['Phase_6_Summ/Test/test.json', 'Phase_6_Summ/Train/train.json']


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

train layout: list 7030 | first record keys: ['id', 'document', 'summary']
train: 7030 records -> 7030 docs | exact duplicates dropped 0 | repeated ids with different content kept 0


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

test: 100 records -> 100 docs | exact duplicates dropped 0 | repeated ids with different content kept 0
dropping 2 train docs with an empty summary

train 6325 | val (held out) 703 | test 100
train document words: median 3124 p90 7817 max 139645 | summary words: min 26 median 637 p90 1487 max 27768 | summary/document ratio median 0.217
val   document words: median 3205 p90 8316 max 90410 | summary words: min 69 median 659 p90 1651 max 8894 | summary/document ratio median 0.216
test  document words: median 3171 p90 8563 max 39695
first test ids: ['4963', '5994', '7130']
train∩test ids 0


In [7]:
# =====================================================================================
# CELL D — [ANCILLARY] tokenisation, local ROUGE / BERTScore scorer, prediction.json writer, baseline
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, os, sys, zipfile, subprocess, importlib
import numpy as np
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
BOS_ID = SPECIAL_TOKENS.index("<s>")     # 2
EOS_ID = SPECIAL_TOKENS.index("</s>")    # 3
NUM_SPECIAL = len(SPECIAL_TOKENS)

def encode_text(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

def ids_to_text(ids):
    """Generated ids -> text: stop at the first </s>, drop special tokens."""
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID)]
    return tok.decode([t for t in ids if t >= NUM_SPECIAL]).strip()

# Tokenise once: d["ids"] = whole judgment, d["sum_ids"] = whole reference summary (train/val)
for name, split in (("train", train), ("val", val), ("test", test)):
    for d in tqdm(split.values(), desc=f"tokenising {name}"):
        d["ids"] = encode_text(d["text"])
        if d["summary"] is not None:
            d["sum_ids"] = encode_text(d["summary"])

# Length statistics in c-1 tokens, to choose WINDOW, MAX_WINDOWS and DEC_LEN in Cell E1
for name, split in tqdm((("train", train), ("val", val), ("test", test)), desc="length stats", leave=False):
    L = np.array([len(d["ids"]) for d in split.values()])
    line = (f"{name:5s} doc tokens: median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
            f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 2048: {(L > 2048).mean():.1%}")
    if split is not test:
        S = np.array([len(d["sum_ids"]) for d in split.values()])
        line += (f" || summary tokens: median {np.median(S):.0f} p90 {np.percentile(S, 90):.0f} "
                 f"p99 {np.percentile(S, 99):.0f} max {S.max()} | > 2048: {(S > 2048).mean():.1%}")
    print(line)
_words = np.array([len(d["summary"].split()) for d in train.values()])
_toks = np.array([len(d["sum_ids"]) for d in train.values()])
print(f"c-1 tokens per summary word (train): {(_toks.sum() / max(1, _words.sum())):.2f}")


# -------------------------------------------------------------------------------------
# Local scorer: BERTScore F1 (primary), ROUGE-1 F1 and ROUGE-L F1 (secondary).
#   The official evaluator's BERTScore model/settings are not stated in the spec. Default here
#   is the bert_score package default for English (roberta-large, no baseline rescaling), the
#   usual choice; change BERTSCORE_MODEL if the TAs say otherwise. Scores are only comparable
#   to Codabench if the settings match, so use them for RELATIVE comparisons between runs.
#   These scorer models are used only to measure, never to train.
# -------------------------------------------------------------------------------------
def _ensure(pkg, pip_name):
    try:
        return importlib.import_module(pkg)
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)
        try:
            return importlib.import_module(pkg)
        except ImportError:
            print(f"  could not install {pip_name} (internet off?)")
            return None

_rouge_mod = _ensure("rouge_score", "rouge-score")
_bs_mod = _ensure("bert_score", "bert-score")
BERTSCORE_MODEL = None     # None = bert_score default for lang="en" (roberta-large)
BERTSCORE_BATCH = 16

if _rouge_mod is not None:
    from rouge_score import rouge_scorer
    _rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def _f(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0

def rouge_pair(pred, ref):
    """-> (ROUGE-1 F1, ROUGE-L F1). rouge_score package if available, else a plain re-implementation."""
    if _rouge_mod is not None:
        s = _rs.score(ref, pred)
        return s["rouge1"].fmeasure, s["rougeL"].fmeasure
    tp = re.findall(r"[a-z0-9]+", pred.lower())
    tr = re.findall(r"[a-z0-9]+", ref.lower())
    if not tp or not tr:
        return 0.0, 0.0
    cp, cr = Counter(tp), Counter(tr)
    ov = sum((cp & cr).values())
    l = _lcs(tp, tr)
    return _f(ov / len(tp), ov / len(tr)), _f(l / len(tp), l / len(tr))

_bs_scorer = None   # BERTScorer, loaded once and reused (bert_score.score() would reload roberta-large every call)

def bertscore_f1(preds, refs, chunk=256):
    """Per-pair BERTScore F1, computed in chunks with a tqdm bar."""
    global _bs_scorer
    if _bs_mod is None:
        return None
    from bert_score import BERTScorer
    if _bs_scorer is None:
        kw = dict(model_type=BERTSCORE_MODEL) if BERTSCORE_MODEL else dict(lang="en")
        _bs_scorer = BERTScorer(batch_size=BERTSCORE_BATCH,
                                device="cuda" if torch.cuda.is_available() else "cpu", **kw)
    preds = [p if p.strip() else "." for p in preds]
    out = []
    for i in tqdm(range(0, len(preds), chunk), desc="BERTScore", leave=False):
        _, _, F1 = _bs_scorer.score(preds[i:i + chunk], refs[i:i + chunk], verbose=False)
        out.append(F1.cpu().numpy())
    torch.cuda.empty_cache()
    return np.concatenate(out)

def evaluate_summaries(pred, docs, show=True, with_bertscore=True):
    """pred: {doc_id: summary str}. Scores every doc in docs that has a reference summary."""
    ids = [k for k in docs if docs[k].get("summary")]
    P = [pred[k] for k in ids]
    R = [docs[k]["summary"] for k in ids]
    r1, rl = zip(*[rouge_pair(p, r) for p, r in tqdm(zip(P, R), total=len(ids), desc="ROUGE", leave=False)])
    bs = bertscore_f1(P, R) if with_bertscore else None
    pw = np.array([len(p.split()) for p in P]); rw = np.array([len(r.split()) for r in R])
    res = {"bertscore_f1": None if bs is None else float(np.mean(bs)),
           "rouge1_f1": float(np.mean(r1)), "rougeL_f1": float(np.mean(rl)),
           "pred_words_median": float(np.median(pw)), "ref_words_median": float(np.median(rw)),
           "len_ratio_median": float(np.median(pw / np.maximum(1, rw))),
           "per_doc": {k: {"r1": a, "rl": b, "bs": None if bs is None else float(c), "pw": int(x), "rw": int(y)}
                       for k, a, b, c, x, y in zip(ids, r1, rl, bs if bs is not None else [None] * len(ids), pw, rw)}}
    if show:
        bstxt = "n/a" if res["bertscore_f1"] is None else f"{res['bertscore_f1']:.4f}"
        print(f"BERTScore F1 {bstxt} | ROUGE-1 F1 {res['rouge1_f1']:.4f} | ROUGE-L F1 {res['rougeL_f1']:.4f} | "
              f"words pred/ref median {res['pred_words_median']:.0f}/{res['ref_words_median']:.0f} "
              f"(ratio {res['len_ratio_median']:.2f}) | n={len(ids)}")
    return res

def repetition_rate(texts, n=3):
    """Share of word n-grams that repeat an earlier n-gram of the same summary (redundancy, for the report)."""
    rates = []
    for t in tqdm(list(texts), desc="repetition", leave=False):
        w = t.lower().split()
        g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        if g:
            rates.append(1 - len(set(g)) / len(g))
    return float(np.mean(rates)) if rates else 0.0

def length_buckets(res, edges=(0.5, 0.8, 1.2, 1.5)):
    """Mean metrics by predicted/reference length ratio (how generation length moves the scores)."""
    rows = list(res["per_doc"].values())
    e = (0.0,) + tuple(edges) + (float("inf"),)
    print(f"{'pred/ref words':>16s} {'n':>5s} {'R-1':>6s} {'R-L':>6s} {'BS':>6s}")
    for lo, hi in zip(e[:-1], e[1:]):
        g = [r for r in rows if lo <= r["pw"] / max(1, r["rw"]) < hi]
        if g:
            bs = [r["bs"] for r in g if r["bs"] is not None]
            print(f"{lo:7.2f}-{hi:<8.2f} {len(g):5d} {np.mean([r['r1'] for r in g]):6.3f} "
                  f"{np.mean([r['rl'] for r in g]):6.3f} {np.mean(bs) if bs else float('nan'):6.3f}")


# -------------------------------------------------------------------------------------
# prediction.json writer: a JSON list of {"id": <test id>, "summary": str}, one per test id,
# ids exactly as in test.json, then submission.zip with prediction.json at the root.
# -------------------------------------------------------------------------------------
def write_prediction(pred, out_dir="/kaggle/working"):
    """pred: {test_id: summary str}."""
    os.makedirs(out_dir, exist_ok=True)
    missing = [k for k in test if k not in pred]
    extra = [k for k in pred if k not in test]
    assert not missing, f"{len(missing)} test ids have no summary, e.g. {missing[:3]}"
    assert not extra, f"{len(extra)} ids are not in test.json, e.g. {extra[:3]}"
    rows = []
    for k in test:
        s = pred[k]
        assert isinstance(s, str), f"{k}: summary must be a string"
        rows.append({"id": k, "summary": s if s.strip() else "."})   # never submit an empty string
    n_empty = sum(not pred[k].strip() for k in test)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    back = json.load(open(path, encoding="utf-8"))
    assert [r["id"] for r in back] == list(test) and len(back) == len(test)
    w = np.array([len(r["summary"].split()) for r in back])
    print(f"wrote {path} and {zpath}: {len(back)} summaries | words min {w.min()} median {np.median(w):.0f} "
          f"max {w.max()} | empty (written as '.') {n_empty}")
    return zpath


# -------------------------------------------------------------------------------------
# Baseline: lead-N words of the document, N = median train summary length in words.
# Any trained model should beat this on val.
# -------------------------------------------------------------------------------------
LEAD_N = int(np.median(_words))
lead_pred = {k: " ".join(d["text"].split()[:LEAD_N]) for k, d in tqdm(val.items(), desc="lead baseline", leave=False)}
print(f"\nlead-{LEAD_N} baseline on val:")
baseline_res = evaluate_summaries(lead_pred, val)


tokenising train:   0%|          | 0/6325 [00:00<?, ?it/s]

tokenising val:   0%|          | 0/703 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/100 [00:00<?, ?it/s]

length stats:   0%|          | 0/3 [00:00<?, ?it/s]

train doc tokens: median 3929 p90 9871 p99 31083 max 173493 | > 2048: 83.7% || summary tokens: median 805 p90 1925 p99 5571 max 35701 | > 2048: 8.6%
val   doc tokens: median 4010 p90 10353 p99 29302 max 110501 | > 2048: 85.1% || summary tokens: median 832 p90 2087 p99 5438 max 11599 | > 2048: 10.5%
test  doc tokens: median 3931 p90 10940 p99 36152 max 48629 | > 2048: 86.0%
c-1 tokens per summary word (train): 1.29
installing bert-score ...


lead baseline:   0%|          | 0/703 [00:00<?, ?it/s]

... (9 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore:   0%|          | 0/3 [00:00<?, ?it/s]

BERTScore F1 0.8311 | ROUGE-1 F1 0.4939 | ROUGE-L F1 0.2473 | words pred/ref median 637/659 (ratio 0.97) | n=703


In [8]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared decoder-only plumbing for the c-1 Phase 6 (summarisation) runs
# Run once after Cells 0, A, B, C, D. Then run Cell E1 (AdaLoRA).
# Needs from earlier cells: lm, cfg, device, features, tok, train, val, test (d["ids"], d["sum_ids"]),
#   PAD_ID, BOS_ID, EOS_ID, UNK_ID, ids_to_text, evaluate_summaries, write_prediction, _bs_mod
#
# c-1 has no encoder, so a summary is generated as a CONTINUATION of the document:
#     input  = [document tokens] [SEP] [summary tokens] </s>
#     loss   = next-token cross-entropy on the summary tokens and the final </s> ONLY
#              (document and SEP positions get target IGNORE, so they cost nothing)
#     decode = feed [document tokens] [SEP], then generate until </s> with a KV cache.
# Everything must fit the c-1 position table: MAX_POS = cfg.max_seq_len = 2048 tokens, so
#     document budget = 2048 - len(SEP) - max(DEC_LEN, MAX_NEW_TOKENS).
#
# Long documents (median ~3,900 tokens, 92% longer than the budget): windows of WINDOW tokens
# every STRIDE tokens (Phase 2-6 make_windows), DOC_REPR picks which of them stand for the
# document ("first", "last", "first_last", "spread", "all"), and the chosen windows are joined
# in document order (overlaps kept once) into the document part. This is the c-6 "concat"
# fusion; "fid" has no decoder-only equivalent. "first" with MAX_WINDOWS 1 and WINDOW = budget
# is plain truncation; "all" is the same as truncation here (cut at the budget).
#
# Batching: training sequences are padded on the RIGHT (causal attention: real tokens never
# see padding, so no mask is needed). Generation prompts are padded on the LEFT with a key
# mask and per-row positions, so every row's real tokens get the same positions and the same
# attention as when run alone. The decoder blocks are the SAME modules with the SAME weights
# (AdaLoRA wrappers included, every linear is called as a module); attention runs in PyTorch's
# fused scaled_dot_product_attention with the Phase 1 scale. Self-checks at the end of the
# cell prove the padded / cached / summary-only paths equal the Phase 1 model.
#
# Also here: the AdaLoRA rank allocator (budget schedule + importance scores + pruning,
# PEFT RankAllocator logic, from your Phase 3 c-5 AdaLoRA cells), as a training hook.
# Nothing in this cell is a model design or a hyperparameter choice: every choice is an
# argument you set in Cell E1.
# =====================================================================================
import math, time, random, os, json, re
import numpy as np
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model          # 1024 for c-1
MAX_POS = cfg.max_seq_len       # 2048: longest input (document + SEP + summary)
IGNORE = -100                   # target id the loss skips
GRAD_CHECKPOINT = False         # set in Cell E1
_AMP = torch.cuda.is_available()


# -------------------------------------------------------------------------------------
# Decoder path: CausalSelfAttention / Block / DecoderLM.forward, same maths, plus optional
# per-row positions, attention mask and KV cache (for batched, cached generation)
# -------------------------------------------------------------------------------------
def _attn(attn, x, mask=None, cache=None):
    """x (B, T, C). mask: None (plain causal) or bool (B, 1, T, S), True = may attend.
    cache: dict holding past keys/values of this layer (generation), or None."""
    B, T, C = x.shape
    H, D = attn.n_heads, attn.d_head
    q, k, v = [p(x).view(B, T, H, D).transpose(1, 2) for p in (attn.q_proj, attn.k_proj, attn.v_proj)]
    if cache is not None:
        if "k" in cache:
            k = torch.cat([cache["k"], k], dim=2)
            v = torch.cat([cache["v"], v], dim=2)
        cache["k"], cache["v"] = k, v
    dp = attn.attn_dropout.p if attn.training else 0.0
    if mask is None:
        causal = T > 1                        # one new token may see every cached position
        if causal and k.size(2) != T:
            raise ValueError("multi-token step with a filled cache needs a mask")
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp, is_causal=causal, scale=attn.attention_scale)
    else:
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, scale=attn.attention_scale)
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

def _block(blk, x, mask=None, cache=None):
    if blk.norm_first:                        # Block.forward, pre-norm
        x = x + _attn(blk.attn, blk.ln1(x), mask, cache)
        return x + blk.ffn(blk.ln2(x))
    x = blk.ln1(x + _attn(blk.attn, x, mask, cache))   # Block.forward, post-norm
    return blk.ln2(x + blk.ffn(x))

def hidden(backbone, idx, pos=None, mask=None, caches=None):
    """idx (B, T) -> last hidden states (B, T, 1024) after ln_f (LM head not applied).
    pos (B, T) positions per token (None = 0..T-1), mask / caches as in _attn."""
    T = idx.size(1)
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        if pos is None:
            assert T <= MAX_POS, f"input of {T} tokens > {MAX_POS}"
            x = x + backbone.pos_emb[:T, :]
        else:
            assert int(pos.max()) < MAX_POS, f"position {int(pos.max())} >= {MAX_POS}"
            x = x + backbone.pos_emb[pos]
    use_ckpt = GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training and caches is None and mask is None
    for i, blk in enumerate(backbone.blocks):
        c = None if caches is None else caches[i]
        x = checkpoint(_block, blk, x, use_reentrant=False) if use_ckpt else _block(blk, x, mask, c)
    return backbone.ln_f(x)


# -------------------------------------------------------------------------------------
# Which linears can be wrapped, by short name (same names in all 8 blocks)
# -------------------------------------------------------------------------------------
_WHERE = {"q_proj": ("attn", "q_proj"), "k_proj": ("attn", "k_proj"), "v_proj": ("attn", "v_proj"),
          "out_proj": ("attn", "out_proj"), "ffn_in": ("ffn", "net", "0"), "ffn_out": ("ffn", "net", "2")}
ALL_TARGETS = tuple(_WHERE)

def model_linears(backbone, which):
    """Yield (parent, attr, module) for each chosen linear, block by block, in `which` order."""
    for w in which:
        assert w in _WHERE, f"unknown target {w}; choose from {ALL_TARGETS}"
    for blk in backbone.blocks:
        for w in which:
            parent = blk
            for p in _WHERE[w][:-1]:
                parent = getattr(parent, p)
            yield parent, _WHERE[w][-1], getattr(parent, _WHERE[w][-1])

def wrap_linears(backbone, which, make_wrapper):
    """Replace each chosen nn.Linear by make_wrapper(linear). Returns how many were wrapped."""
    todo = list(model_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Long document -> document part of the prompt (Phase 2-6 make_windows / select_windows)
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]

def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)

def doc_part(ids, window, stride, doc_repr, max_windows, budget):
    """Chosen windows joined in document order (overlapping parts kept once), cut to budget."""
    out, last_end = [], 0
    for s, e in select_windows(make_windows(len(ids), window, stride), doc_repr, max_windows):
        s = max(s, last_end)
        out += ids[s:e]
        last_end = e
    return out[:budget]

def doc_budget(sep_ids, dec_len, max_new_tokens):
    return MAX_POS - len(sep_ids) - max(dec_len, max_new_tokens)

def build_examples(docs, window, stride, doc_repr, max_windows, dec_len, max_new_tokens, sep_ids):
    """One example per document: {"doc", "prompt" (document part + SEP), "n_doc", "summary"
    (summary tokens cut to dec_len - 1, + </s>; None for test)}."""
    budget = doc_budget(sep_ids, dec_len, max_new_tokens)
    assert budget > 0, "SEP + DEC_LEN / MAX_NEW_TOKENS leave no room for the document"
    assert stride <= window and doc_repr in DOC_REPRS and (doc_repr == "all" or max_windows)
    assert doc_repr in ("all", "first") or window * max_windows <= budget, (
        f"MAX_WINDOWS x WINDOW = {window * max_windows} > document budget {budget} "
        f"(= {MAX_POS} - len(SEP) {len(sep_ids)} - max(DEC_LEN, MAX_NEW_TOKENS)); later windows would be cut off")
    ex, n_cut = [], 0
    for k, d in tqdm(docs.items(), desc="examples", leave=False):
        dp_ = doc_part(d["ids"], window, stride, doc_repr, max_windows, budget)
        e = {"doc": k, "prompt": dp_ + list(sep_ids), "n_doc": len(dp_), "summary": None}
        if d.get("sum_ids") is not None:
            e["summary"] = d["sum_ids"][:dec_len - 1] + [EOS_ID]
            n_cut += len(d["sum_ids"]) > dec_len - 1
        ex.append(e)
    nt = np.array([e["n_doc"] for e in ex])
    nd = np.array([len(d["ids"]) for d in docs.values()])
    line = (f"{len(ex)} docs | document budget {budget} tokens | document tokens kept per doc median "
            f"{np.median(nt):.0f} (whole doc {np.median(nd):.0f}) | doc tokens seen {nt.sum() / nd.sum():.1%}")
    if ex[0]["summary"] is not None:
        line += f" | summaries cut at DEC_LEN: {n_cut} ({n_cut / len(ex):.1%})"
    print(line)
    return ex

def _len(e):
    return len(e["prompt"]) + len(e["summary"] or [])

def make_batches(examples, batch_size, shuffle):
    """Similar lengths go together (less padding); batch order shuffled for training."""
    order = sorted(range(len(examples)), key=lambda i: _len(examples[i]))
    batches = [[examples[i] for i in order[s:s + batch_size]] for s in range(0, len(order), batch_size)]
    if shuffle:
        random.shuffle(batches)
    return batches

def to_tensors(batch):
    """Training / val loss batch, right-padded: inp = seq[:-1], target = seq[1:] on summary positions only."""
    inps, tgts = [], []
    for e in batch:
        seq = e["prompt"] + e["summary"]
        inps.append(seq[:-1])
        tgts.append([IGNORE] * (len(e["prompt"]) - 1) + e["summary"])   # position of the last SEP token predicts summary[0]
    T = max(map(len, inps))
    return {"inp": torch.tensor([r + [PAD_ID] * (T - len(r)) for r in inps], dtype=torch.long, device=device),
            "target": torch.tensor([r + [IGNORE] * (T - len(r)) for r in tgts], dtype=torch.long, device=device)}


class DecoderSummModel(nn.Module):
    """c-1 decoder for summarisation. forward(b) -> (logits on summary positions only (N, V), targets (N,)).
    The LM head (tied to tok_emb) is applied only where there is a target, which saves memory."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, b):
        h = hidden(self.backbone, b["inp"])
        sel = b["target"] != IGNORE
        return self.backbone.head(h[sel]), b["target"][sel]


def summ_loss(logits, tgt, label_smoothing=0.0):
    return F.cross_entropy(logits.float(), tgt, label_smoothing=label_smoothing or 0.0)

def _split(batch, micro):
    if not micro or micro >= len(batch):
        return [batch]
    return [batch[i:i + micro] for i in range(0, len(batch), micro)]

@torch.no_grad()
def val_loss(model, examples, batch_size):
    """Mean token cross-entropy (no label smoothing) on the held-out summaries -> (loss, perplexity)."""
    model.eval()
    tot, n = 0.0, 0
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="val loss", leave=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=_AMP):
            logits, tgt = model(to_tensors(batch))
        tot += F.cross_entropy(logits.float(), tgt, reduction="sum").item()
        n += tgt.numel()
    l = tot / max(1, n)
    return l, math.exp(min(l, 50))


# -------------------------------------------------------------------------------------
# Generation: prompts left-padded (key mask + per-row positions), one prefill pass fills the
# KV cache, then one step per new token. Decoding controls (all set in Cell E1):
#   max_new_tokens, min_new_tokens (</s> blocked before this), do_sample (False = greedy),
#   temperature / top_k / top_p (sampling only), repetition_penalty (1.0 = off),
#   no_repeat_ngram (0 = off), max_words (cut the decoded text to this many words, None = off)
# -------------------------------------------------------------------------------------
def _no_repeat(logits, gen, n):
    t = gen.size(1)
    if n < 2 or t < n - 1:
        return logits
    for r in range(gen.size(0)):
        seq = gen[r].tolist()
        tail = tuple(seq[t - (n - 1):])
        banned = {seq[i + n - 1] for i in range(t - n + 1) if tuple(seq[i:i + n - 1]) == tail}
        if banned:
            logits[r, list(banned)] = float("-inf")
    return logits

def _pick(logits, do_sample, temperature, top_k, top_p):
    if not do_sample:
        return logits.argmax(-1, keepdim=True)
    if top_k:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    probs = F.softmax(logits / max(temperature or 1.0, 1e-5), dim=-1)
    if top_p is not None and top_p < 1.0:
        sp, si = torch.sort(probs, descending=True, dim=-1)
        sp = sp.masked_fill(sp.cumsum(-1) - sp > top_p, 0.0)
        sp = sp / sp.sum(-1, keepdim=True)
        return si.gather(-1, torch.multinomial(sp, 1))
    return torch.multinomial(probs, 1)

def _prefill(bb, prompts, caches):
    """Left-pad prompts, run them once, fill caches. -> (last-position logits (B, V), key_mask, n_real)."""
    P = max(map(len, prompts))
    idx = torch.tensor([[PAD_ID] * (P - len(p)) + p for p in prompts], dtype=torch.long, device=device)
    key_mask = idx.new_tensor([[False] * (P - len(p)) + [True] * len(p) for p in prompts], dtype=torch.bool)
    n_real = key_mask.sum(1)
    if bool(key_mask.all()):
        pos, m = None, None
    else:
        pos = (key_mask.long().cumsum(1) - 1).clamp(min=0)
        causal = torch.tril(torch.ones(P, P, dtype=torch.bool, device=device))
        m = (causal[None] & key_mask[:, None, :]) | torch.eye(P, dtype=torch.bool, device=device)[None]
        m = m[:, None]                                  # (B, 1, P, P); pad rows see only themselves (no NaN)
    h = hidden(bb, idx, pos, m, caches)
    return bb.head(h[:, -1]).float(), key_mask, n_real

@torch.no_grad()
def generate(model, batch, max_new_tokens, min_new_tokens=0, do_sample=False, temperature=1.0, top_k=None,
             top_p=None, repetition_penalty=1.0, no_repeat_ngram=0):
    """batch: list of examples. -> list of generated token lists (cut at </s>)."""
    model.eval()
    bb = model.backbone
    prompts = [e["prompt"] for e in batch]
    assert max(map(len, prompts)) + max_new_tokens - 1 <= MAX_POS, "prompt + MAX_NEW_TOKENS > 2048 positions"
    caches = [dict() for _ in bb.blocks]
    with torch.autocast("cuda", dtype=torch.float16, enabled=_AMP):
        logits, key_mask, n_real = _prefill(bb, prompts, caches)
    B = len(prompts)
    gen = torch.zeros((B, 0), dtype=torch.long, device=device)
    done = torch.zeros(B, dtype=torch.bool, device=device)
    padded = not bool(key_mask.all())
    for t in range(max_new_tokens):
        logits[:, [PAD_ID, UNK_ID, BOS_ID]] = float("-inf")
        if t < (min_new_tokens or 0):
            logits[:, EOS_ID] = float("-inf")
        if repetition_penalty and repetition_penalty != 1.0 and gen.size(1):
            seen = torch.zeros_like(logits, dtype=torch.bool).scatter_(1, gen, True)
            seen[:, EOS_ID] = False
            pen = torch.where(logits > 0, logits / repetition_penalty, logits * repetition_penalty)
            logits = torch.where(seen, pen, logits)
        logits = _no_repeat(logits, gen, no_repeat_ngram or 0)
        nxt = _pick(logits, do_sample, temperature, top_k, top_p)
        nxt = torch.where(done[:, None], torch.full_like(nxt, EOS_ID), nxt)
        gen = torch.cat([gen, nxt], dim=1)
        done |= nxt[:, 0] == EOS_ID
        if bool(done.all()) or t == max_new_tokens - 1:
            break
        key_mask = torch.cat([key_mask, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
        pos = (n_real + t)[:, None]
        m = key_mask[:, None, None, :] if padded else None
        with torch.autocast("cuda", dtype=torch.float16, enabled=_AMP):
            logits = bb.head(hidden(bb, nxt, pos, m, caches)[:, -1]).float()
    out = []
    for row in gen.tolist():
        out.append(row[:row.index(EOS_ID)] if EOS_ID in row else row)
    return out

def predict(model, examples, gen_batch_size, gen_kw, max_words=None, desc="generate"):
    """-> {doc_id: summary text}."""
    pred = {}
    t0 = time.time()
    for batch in tqdm(make_batches(examples, gen_batch_size, shuffle=False), desc=desc):
        for e, ids in zip(batch, generate(model, batch, **gen_kw)):
            txt = ids_to_text(ids)
            pred[e["doc"]] = " ".join(txt.split()[:max_words]) if max_words else txt
    print(f"{desc}: {len(pred)} docs in {(time.time() - t0) / 60:.1f} min")
    return pred


# -------------------------------------------------------------------------------------
# AdaLoRA rank allocator (your Phase 3 c-5 AdaLoRAAllocator, PEFT RankAllocator logic), as a
# training hook. Works with any layer that has ada_P (out, r), ada_E (r,), ada_Q (r, in),
# an ada_mask buffer (r,) and orth_penalty().
#   - importance of every P / E / Q entry: EMA of |theta * grad| (beta1) times the EMA of its
#     deviation (uncertainty, beta2); a triplet's score = E score + mean P column + mean Q row
#   - budget (total kept triplets over ALL layers): init_rank x layers for the first tinit
#     steps, cubic decrease to target_rank x layers, then fixed for the last tfinal steps
#   - every delta_t steps (and every step in the final phase): keep the top-`budget` triplets
#     over all layers, E = 0 for the rest (P, Q kept, so a triplet can come back later);
#     in the final phase the pattern is frozen
#   - extra_loss() = orth_reg x mean orthogonality penalty, added to the training loss
# Changes from Phase 3: scores are taken from UNSCALED gradients (train_model unscales the
# fp16 GradScaler before calling the hook), and the state is saved with last_checkpoint.pt.
# -------------------------------------------------------------------------------------
class AdaLoRAAllocator:
    def __init__(self, layers, init_rank, target_rank, total_steps, tinit, tfinal, delta_t, beta1, beta2, orth_reg):
        assert tinit < total_steps - tfinal, "warm-up and final phase leave no steps for pruning"
        self.layers = layers
        self.init_budget, self.target_budget = init_rank * len(layers), target_rank * len(layers)
        self.T, self.tinit, self.tfinal, self.dT = total_steps, tinit, tfinal, delta_t
        self.b1, self.b2, self.orth_reg = beta1, beta2, orth_reg
        self.ipt, self.unc = {}, {}
        self.step, self.budget, self.frozen = 0, self.init_budget, False

    def _params(self, m):
        return {"P": m.ada_P, "E": m.ada_E, "Q": m.ada_Q}

    @torch.no_grad()
    def after_backward(self):
        if self.step >= self.T - self.tfinal:
            return
        for m in self.layers:
            for p in self._params(m).values():
                if p.grad is None or not torch.isfinite(p.grad).all():
                    return                       # fp16 overflow step: GradScaler skips it, so do we
        for li, m in enumerate(self.layers):
            for k, p in self._params(m).items():
                s = (p * p.grad).abs().float()
                key = (li, k)
                if key not in self.ipt:
                    self.ipt[key], self.unc[key] = torch.zeros_like(s), torch.zeros_like(s)
                self.ipt[key].mul_(self.b1).add_((1 - self.b1) * s)
                self.unc[key].mul_(self.b2).add_((1 - self.b2) * (s - self.ipt[key]).abs())

    def _triplet_scores(self, li):
        sc = {k: self.ipt[(li, k)] * self.unc[(li, k)] for k in ("P", "E", "Q")}
        return sc["E"] + sc["P"].mean(0) + sc["Q"].mean(1)   # (rank,)

    def _schedule(self):
        t = self.step
        if t <= self.tinit:
            return self.init_budget, False
        if t > self.T - self.tfinal:
            return self.target_budget, True
        mul = 1 - (t - self.tinit) / (self.T - self.tfinal - self.tinit)
        b = int((self.init_budget - self.target_budget) * mul ** 3 + self.target_budget)
        return b, t % self.dT == 0

    @torch.no_grad()
    def after_step(self):
        self.step += 1
        self.budget, do_mask = self._schedule()
        if not do_mask or not self.ipt:
            return
        if not self.frozen:
            scores = [self._triplet_scores(li) for li in range(len(self.layers))]
            flat = torch.cat(scores)
            keep = torch.zeros_like(flat, dtype=torch.bool)
            keep[flat.topk(min(self.budget, flat.numel())).indices] = True
            for m, kp in zip(self.layers, keep.split([s.numel() for s in scores])):
                m.ada_mask.copy_(kp.float())
            if self.step > self.T - self.tfinal:
                self.frozen = True
        for m in self.layers:
            m.ada_E.mul_(m.ada_mask)

    def extra_loss(self):
        return self.orth_reg * sum(m.orth_penalty() for m in self.layers) / len(self.layers)

    def postfix(self):
        return {"budget": self.budget}

    def rank_pattern(self):
        """Ranks actually used per layer = non-zero E entries (works after loading saved weights).
        Between re-allocation steps a pruned triplet can get a small non-zero E again from the
        optimizer (same as PEFT); in the final phase E is re-masked every step."""
        return [int((m.ada_E != 0).sum()) for m in self.layers]

    def mask_pattern(self):
        """Ranks per layer the allocator last decided (ada_mask), i.e. the allocation itself."""
        return [int(m.ada_mask.sum()) for m in self.layers]

    def state_dict(self):
        return {"ipt": {k: v.cpu() for k, v in self.ipt.items()}, "unc": {k: v.cpu() for k, v in self.unc.items()},
                "step": self.step, "budget": self.budget, "frozen": self.frozen,
                "masks": [m.ada_mask.cpu() for m in self.layers]}

    def load_state_dict(self, s):
        dev = self.layers[0].ada_E.device
        self.ipt = {k: v.to(dev) for k, v in s["ipt"].items()}
        self.unc = {k: v.to(dev) for k, v in s["unc"].items()}
        self.step, self.budget, self.frozen = s["step"], s["budget"], s["frozen"]
        for m, mk in zip(self.layers, s["masks"]):
            m.ada_mask.copy_(mk.to(dev))


# -------------------------------------------------------------------------------------
# Training: AdamW on the trainable parameters only (fp16 autocast + GradScaler), optional
# micro-batching (same update, less memory), grad clipping, per-epoch time cap, hooks
# (the AdaLoRA allocator). Best epoch chosen by select_by on val:
#   "loss" = val token cross-entropy (cheap, no generation)
#   "rougeL" / "rouge1" / "bertscore" = generate val_gen_n held-out docs each epoch and score them
# Saves best_trainable.pt (trainable tensors of the best epoch) and last_checkpoint.pt
# (everything needed to resume: trainable, best, optimizer, scaler, scheduler, hook state, epoch, log).
# -------------------------------------------------------------------------------------
def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_model(model, train_ex, val_ex, val_docs, lr, epochs, batch_size, weight_decay, grad_clip=None,
                label_smoothing=0.0, lr_lambda=None, save_path="best_trainable.pt", last_path=None,
                resume_from=None, micro_batch=None, max_epoch_minutes=None, select_by="loss",
                val_gen_n=None, gen_batch_size=8, gen_kw=None, max_words=None, hooks=None):
    assert select_by in ("loss", "rougeL", "rouge1", "bertscore")
    assert select_by != "bertscore" or _bs_mod is not None, "bert-score is not installed: pick another SELECT_BY"
    params = [p for p in model.parameters() if p.requires_grad]
    assert params, "no trainable parameters"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=_AMP)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    best, log, start = -float("inf"), [], 0
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"):
            sched.load_state_dict(ck["sched"])
        if hooks is not None and ck.get("hooks") is not None:
            hooks.load_state_dict(ck["hooks"])
        best, log, start = ck["best"], ck["log"], ck["epoch"]
        torch.save(ck["best_trainable"], save_path)
        print(f"resumed after epoch {start} (best {select_by} {abs(best):.4f})")
        del ck
    gen_ids = set(list(val_docs)[:val_gen_n]) if (select_by != "loss" and val_gen_n) else set(val_docs)
    gen_ex = [e for e in val_ex if e["doc"] in gen_ids]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start, epochs):
        model.train()
        t0, tot, ntok, seen = time.time(), 0.0, 0, 0
        batches = make_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {ep + 1}/{epochs}")
        for batch in pbar:
            opt.zero_grad(set_to_none=True)
            parts = _split(batch, micro_batch)
            n_all = sum(len(e["summary"]) for e in batch)
            for part in parts:
                b = to_tensors(part)
                with torch.autocast("cuda", dtype=torch.float16, enabled=_AMP):
                    logits, tgt = model(b)
                n_part = tgt.numel()
                loss = summ_loss(logits, tgt, label_smoothing)
                full = loss * n_part / n_all                       # token-weighted, same as one big batch
                if hooks is not None:
                    full = full + hooks.extra_loss() * n_part / n_all
                scaler.scale(full).backward()
                tot += loss.item() * n_part; ntok += n_part
            if grad_clip or hooks is not None:
                scaler.unscale_(opt)                               # true gradients for clipping and AdaLoRA scores
            if hooks is not None:
                hooks.after_backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if hooks is not None:
                hooks.after_step()
            if sched:
                sched.step()
            seen += 1
            pf = {"loss": f"{tot / max(1, ntok):.4f}", "lr": f"{opt.param_groups[0]['lr']:.2e}"}
            if hooks is not None:
                pf.update(hooks.postfix())
            pbar.set_postfix(pf)
            if max_epoch_minutes and time.time() - t0 > 60 * max_epoch_minutes:
                print(f"  time cap: stopped epoch after {seen}/{len(batches)} batches")
                break
        vl, ppl = val_loss(model, val_ex, batch_size)
        row = {"epoch": ep + 1, "train_loss": tot / max(1, ntok), "val_loss": vl, "val_ppl": ppl,
               "batches": seen, "minutes": (time.time() - t0) / 60}
        if hooks is not None:
            row.update({"rank_budget": hooks.budget, "ranks_kept": sum(hooks.rank_pattern())})
        if select_by == "loss":
            score = -vl
        else:
            p = predict(model, gen_ex, gen_batch_size, gen_kw, max_words, desc="val generate")
            r = evaluate_summaries(p, {k: val_docs[k] for k in p}, show=True,
                                   with_bertscore=select_by == "bertscore")
            row.update({k: r[k] for k in ("rouge1_f1", "rougeL_f1", "bertscore_f1", "pred_words_median")})
            score = {"rougeL": r["rougeL_f1"], "rouge1": r["rouge1_f1"], "bertscore": r["bertscore_f1"]}[select_by]
        log.append(row)
        improved = score > best
        if improved:
            best = score
            torch.save(trainable_state(model), save_path)
        print(f"epoch {ep + 1}: train loss {row['train_loss']:.4f} | val loss {vl:.4f} (ppl {ppl:.1f}) | "
              f"{select_by} {abs(score):.4f}{' *best*' if improved else ''} | {row['minutes']:.1f} min"
              + (f" | rank budget {hooks.budget}, kept {row['ranks_kept']}" if hooks is not None else ""))
        if last_path:
            torch.save({"trainable": trainable_state(model), "best_trainable": torch.load(save_path),
                        "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None,
                        "hooks": hooks.state_dict() if hooks is not None else None,
                        "epoch": ep + 1, "best": best, "log": log}, last_path)
    return log, best


# -------------------------------------------------------------------------------------
# Long-document analysis helpers (for the report)
# -------------------------------------------------------------------------------------
_WORD = re.compile(r"[a-z][a-z0-9]{3,}")

def coverage_report(docs, examples):
    """How much of each reference summary's vocabulary (words of 4+ letters) the model can SEE:
    in the document part of the prompt vs in the whole document. The gap is information the
    long-document strategy throws away before the model even starts."""
    by = {e["doc"]: e for e in examples}
    cov_in, cov_doc = [], []
    for k, d in docs.items():
        if not d.get("summary") or k not in by:
            continue
        sw = set(_WORD.findall(d["summary"].lower()))
        if not sw:
            continue
        seen = set(_WORD.findall(tok.decode(by[k]["prompt"][:by[k]["n_doc"]]).lower()))
        full = set(_WORD.findall(d["text"].lower()))
        cov_in.append(len(sw & seen) / len(sw)); cov_doc.append(len(sw & full) / len(sw))
    print(f"summary words found in the model input {np.mean(cov_in):.1%} vs in the whole document "
          f"{np.mean(cov_doc):.1%} (n={len(cov_in)})")
    return float(np.mean(cov_in)), float(np.mean(cov_doc))

def show_examples(pred, docs, n=2, words=120):
    for k in list(pred)[:n]:
        print(f"--- {k}\nPRED: {' '.join(pred[k].split()[:words])}")
        if docs[k].get("summary"):
            print(f"REF : {' '.join(docs[k]['summary'].split()[:words])}")


# -------------------------------------------------------------------------------------
# Self-checks: the fused / padded / cached / summary-only paths equal the Phase 1 model
# -------------------------------------------------------------------------------------
def _self_checks():
    lm.eval()
    d = next(iter(train.values()))
    ids = d["ids"][:48]
    x = torch.tensor([ids], device=device)
    with torch.no_grad():
        ref = lm(x).float()                                           # Phase 1 DecoderLM.forward
        e1 = (lm.head(hidden(lm, x)).float() - ref).abs().max().item()
        e2 = (lm.head(features(x)).float() - ref).abs().max().item()  # Cell B path, for reference
        # left-padded prompt in a batch with a longer one: same last-position logits as alone
        short = ids[:30]
        caches = [dict() for _ in lm.blocks]
        lg, km, nr = _prefill(lm, [short, ids], caches)
        e3 = (lg[0] - lm(torch.tensor([short], device=device))[0, -1].float()).abs().max().item()
        e4 = (lg[1] - ref[0, -1]).abs().max().item()
        # cached step decoding (with the padding mask) vs the full forward
        cont = d["ids"][48:56]
        full_short = lm(torch.tensor([short + cont], device=device))[0].float()
        km2, err = km, 0.0
        for t, tk in enumerate(cont):
            km2 = torch.cat([km2, torch.ones(2, 1, dtype=torch.bool, device=device)], dim=1)
            step = torch.tensor([[tk], [tk]], device=device)
            out = lm.head(hidden(lm, step, (nr + t)[:, None], km2[:, None, None, :], caches)[:, -1]).float()
            err = max(err, (out[0] - full_short[len(short) + t]).abs().max().item())
        # summary-only logits used by the loss = the full model's logits at those positions
        e = {"prompt": ids[:20] + [EOS_ID], "summary": ids[20:30] + [EOS_ID]}
        b = to_tensors([e, {"prompt": ids[:12] + [EOS_ID], "summary": ids[12:16] + [EOS_ID]}])
        lg5, tg5 = DecoderSummModel(lm)(b)
        full = lm(b["inp"][:1]).float()[0, 20:31]
        e5 = (lg5[:11].float() - full).abs().max().item()
        assert tg5[:11].tolist() == e["summary"]
    print(f"self-checks max |diff|: fused decoder {e1:.1e} | Cell B features {e2:.1e} | left-padded prompt "
          f"{e3:.1e} / unpadded {e4:.1e} | cached decoding {err:.1e} | summary-only logits {e5:.1e}  (all ~0)")
    assert max(e1, e2, e3, e4, err, e5) < 1e-3, "padded / cached forward differs from the Phase 1 model"

_self_checks()


self-checks max |diff|: fused decoder 6.1e-05 | Cell B features 0.0e+00 | left-padded prompt 5.3e-05 / unpadded 9.9e-05 | cached decoding 1.2e-04 | summary-only logits 3.8e-05  (all ~0)


In [9]:
# =====================================================================================
# CELL E1 — AdaLoRA on the c-1 decoder for Phase 6 abstractive summarisation
#
# AdaLoRA (Zhang et al., ICLR 2023): each adapted linear becomes  W x + P diag(E) Q x,
# an SVD-shaped update made of rank-1 triplets (P[:, i], E[i], Q[i, :]). During training every
# triplet gets an importance score, and the least important triplets are pruned (E[i] = 0)
# across ALL adapted layers together, so the layers that matter more keep more rank. The total
# rank budget falls on a cubic schedule from ADA_INIT_RANK x layers to ADA_TARGET_RANK x layers.
#
# Input format (decoder-only): [document part] [SEP] [summary] </s>, loss on summary + </s> only.
#
# Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# (wrapping replaces the linears in place, and wrap_linears refuses to wrap twice).
#
# >>> FILL IN: every value marked  # <<< FILL  in section [H]. <<<
# Values marked "c-6" are copied from your Phase 6 c-6 LORMA run. The c-1 input format and
# window sizes (the c-6 ones do not fit 2048 positions) and every AdaLoRA-specific value are
# left None on purpose. Sections marked [ANCILLARY] are plumbing.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — YOURS
# -------------------------------------------------------------------------------------
# input format
SEP = "eos"

# long document -> document part of the prompt
WINDOW      = 512
STRIDE      = 256
DOC_REPR    = "spread"
MAX_WINDOWS = 3
DEC_LEN     = 256

# AdaLoRA: where, and the rank budget
ADA_TARGETS = ("q_proj", "k_proj", "v_proj", "out_proj", "ffn_in", "ffn_out")

ADA_INIT_RANK   = 12
ADA_TARGET_RANK = 8

# AdaLoRA: budget schedule
TINIT_FRAC  = 0.1
TFINAL_FRAC = 0.1
DELTA_T     = 10
BETA1       = 0.85
BETA2       = 0.85
ORTH_REG    = 0.1

# training   (same as c-6 LORMA)
LR              = 1e-4
EPOCHS          = 10
BATCH_SIZE      = 4
WEIGHT_DECAY    = 0.01
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.05
LR_LAMBDA       = None
MAX_EPOCH_MINUTES = None
SELECT_BY       = "loss"
VAL_GEN_N       = None

# decoding
GEN_BATCH_SIZE     = 4
MAX_NEW_TOKENS     = 220
MIN_NEW_TOKENS     = 20
DO_SAMPLE          = False
TEMPERATURE        = 1.0
TOP_K              = None
TOP_P              = 1.0
REPETITION_PENALTY = 1.1
NO_REPEAT_NGRAM    = 3
MAX_SUMMARY_WORDS  = 150

SEED = 42

# [ANCILLARY] GPU memory savers
GRAD_CHECKPOINT = False
MICRO_BATCH     = 1

# [ANCILLARY] continue an earlier run
RESUME_FROM = None

# [ANCILLARY] generate summaries for TRAIN documents
DUMP_TRAIN_SUMMARIES = False

_required = ["SEP", "WINDOW", "STRIDE", "DOC_REPR", "DEC_LEN", "ADA_TARGETS", "ADA_INIT_RANK", "ADA_TARGET_RANK",
             "TINIT_FRAC", "TFINAL_FRAC", "DELTA_T", "BETA1", "BETA2", "ORTH_REG", "LR", "EPOCHS", "BATCH_SIZE",
             "WEIGHT_DECAY", "GRAD_CLIP", "LABEL_SMOOTHING", "SELECT_BY", "GEN_BATCH_SIZE", "MAX_NEW_TOKENS",
             "MIN_NEW_TOKENS", "DO_SAMPLE", "REPETITION_PENALTY", "NO_REPEAT_NGRAM", "SEED"]
if DO_SAMPLE:
    _required += ["TEMPERATURE", "TOP_P"]
if DOC_REPR != "all":
    _required += ["MAX_WINDOWS"]
_unset = [n for n in _required if globals()[n] is None]
assert not _unset, f"fill in section [H] first: {_unset}"
assert isinstance(ADA_TARGETS, (tuple, list)) and ADA_TARGETS and all(t in ALL_TARGETS for t in ADA_TARGETS), \
    f"ADA_TARGETS: non-empty tuple from {ALL_TARGETS}"
assert 0 < ADA_TARGET_RANK <= ADA_INIT_RANK, "need 0 < ADA_TARGET_RANK <= ADA_INIT_RANK"
assert 0 <= TINIT_FRAC and 0 <= TFINAL_FRAC and TINIT_FRAC + TFINAL_FRAC < 1, "TINIT_FRAC + TFINAL_FRAC must be < 1"
assert DELTA_T >= 1 and 0 <= BETA1 < 1 and 0 <= BETA2 < 1 and ORTH_REG >= 0
assert STRIDE <= WINDOW and MAX_NEW_TOKENS <= MAX_POS and DEC_LEN <= MAX_POS


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
SEP_IDS = [EOS_ID] if SEP == "eos" else encode_text(SEP)
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS, dec_len=DEC_LEN,
             max_new_tokens=MAX_NEW_TOKENS, sep_ids=SEP_IDS)
GEN_KW = dict(max_new_tokens=MAX_NEW_TOKENS, min_new_tokens=MIN_NEW_TOKENS, do_sample=DO_SAMPLE,
              temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
              repetition_penalty=REPETITION_PENALTY, no_repeat_ngram=NO_REPEAT_NGRAM)
print(f"SEP = {SEP!r} -> ids {SEP_IDS} | document budget {doc_budget(SEP_IDS, DEC_LEN, MAX_NEW_TOKENS)} tokens")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c1_adalora_summ_{DOC_REPR}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
LAST = f"{OUT_DIR}/last_checkpoint.pt"

print("AdaLoRA target names (same in all blocks):")
for _name in ALL_TARGETS:
    _parent, _attr, _lin = next(model_linears(lm, (_name,)))
    print(f"  {_name:9s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")


# -------------------------------------------------------------------------------------
# [L] AdaLoRA layer: base(x) + P diag(E) Q x  (your Phase 3 c-5 AdaLoRALinear, unchanged)
# -------------------------------------------------------------------------------------
class AdaLoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base                         # frozen pretrained layer (W, b)
        for p in self.base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.rank = rank
        # Triplet i = (ada_P[:, i], ada_E[i], ada_Q[i, :]). E plays the singular values.
        # E starts at 0, so P diag(E) Q = 0 and the layer is exactly the base layer at step 0.
        self.ada_P = nn.Parameter(torch.randn(self.out_features, rank) * 0.02)
        self.ada_E = nn.Parameter(torch.zeros(rank))
        self.ada_Q = nn.Parameter(torch.randn(rank, self.in_features) * 0.02)
        self.register_buffer("ada_mask", torch.ones(rank))   # 1 = triplet kept, 0 = pruned

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.ada_Q) * self.ada_E, self.ada_P)

    def orth_penalty(self):
        I = torch.eye(self.rank, device=self.ada_P.device)
        return (torch.linalg.norm(self.ada_P.T @ self.ada_P - I, "fro")
                + torch.linalg.norm(self.ada_Q @ self.ada_Q.T - I, "fro"))


# [ANCILLARY] quick check on a dummy linear: shape, and that it really is W x + b + P diag(E) Q x
_t = AdaLoRALinear(nn.Linear(64, 96), rank=ADA_INIT_RANK)
_x = torch.randn(2, 5, 64)
with torch.no_grad():
    assert _t(_x).shape == (2, 5, 96) and torch.allclose(_t(_x), _t.base(_x))      # E = 0 -> base layer
    nn.init.normal_(_t.ada_E, std=0.5)
    _dW = _t.ada_P @ torch.diag(_t.ada_E) @ _t.ada_Q
    assert torch.allclose(_t(_x), _x @ (_t.base.weight + _dW).T + _t.base.bias, atol=1e-5), "layer is not W + P diag(E) Q"
print("AdaLoRALinear = W + P diag(E) Q checked")
del _t, _x, _dW


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze c-1, wrap the chosen linears, check identity at init
# -------------------------------------------------------------------------------------
lm.eval()
_d = next(iter(train.values()))
probe = torch.tensor([_d["ids"][:64]], device=device)
with torch.no_grad():
    ref = lm(probe).float()

for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(lm, ADA_TARGETS, lambda lin: AdaLoRALinear(lin, ADA_INIT_RANK).to(device))
ada_layers = [m for _, _, m in model_linears(lm, ADA_TARGETS)]          # block by block, in ADA_TARGETS order
ada_names = [f"block{i}.{w}" for i in range(cfg.n_layers) for w in ADA_TARGETS]
assert all(isinstance(m, AdaLoRALinear) for m in ada_layers) and len(ada_layers) == n_wrapped

with torch.no_grad():
    diff = (lm.head(hidden(lm, probe)).float() - ref).abs().max().item()
print(f"wrapped {n_wrapped} linears with AdaLoRA | identity-at-init max |diff| = {diff:.2e} (should be ~0)")
assert diff < 1e-3, "wrapped model differs from the original at init"

model = DecoderSummModel(lm).to(device)
count_params(model)
_other = [n for n, p in model.named_parameters() if p.requires_grad and not n.split(".")[-1].startswith("ada_")]
assert not _other, f"only AdaLoRA P/E/Q should train, found {_other[:3]}"


# -------------------------------------------------------------------------------------
# [ANCILLARY] data, rank allocator, training, val scoring, rank table, test submission
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
val_ex = build_examples(val, **BUILD)
test_ex = build_examples(test, **BUILD)
print("\n-- long-document strategy: what the model sees (val)")
cov = coverage_report(val, val_ex)

steps_per_epoch = len(make_batches(train_ex, BATCH_SIZE, shuffle=False))
TOTAL_STEPS = steps_per_epoch * EPOCHS
allocator = AdaLoRAAllocator(
    ada_layers, ADA_INIT_RANK, ADA_TARGET_RANK, TOTAL_STEPS,
    tinit=int(TINIT_FRAC * TOTAL_STEPS), tfinal=int(TFINAL_FRAC * TOTAL_STEPS),
    delta_t=DELTA_T, beta1=BETA1, beta2=BETA2, orth_reg=ORTH_REG)
print(f"\n{steps_per_epoch} steps/epoch, {TOTAL_STEPS} total | rank budget {allocator.init_budget} -> "
      f"{allocator.target_budget} triplets over {len(ada_layers)} layers | no pruning for steps 1-{allocator.tinit}, "
      f"cubic pruning every {DELTA_T} steps, ranks frozen after step {TOTAL_STEPS - allocator.tfinal}")
if MAX_EPOCH_MINUTES:
    print("NOTE: MAX_EPOCH_MINUTES is set. The schedule assumes full epochs, so if epochs are cut short the "
          "budget never reaches the target / the final phase. Check 'rank budget' in the epoch lines.")

t_train = time.time()
log, best = train_model(
    model, train_ex, val_ex, val,
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    label_smoothing=LABEL_SMOOTHING, lr_lambda=LR_LAMBDA, save_path=SAVE, last_path=LAST,
    resume_from=RESUME_FROM, micro_batch=MICRO_BATCH, max_epoch_minutes=MAX_EPOCH_MINUTES,
    select_by=SELECT_BY, val_gen_n=VAL_GEN_N, gen_batch_size=GEN_BATCH_SIZE, gen_kw=GEN_KW,
    max_words=MAX_SUMMARY_WORDS, hooks=allocator)
train_min = (time.time() - t_train) / 60
train_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0

# Where AdaLoRA put the rank: one row per block, one column per target
def _rank_table(title, rk):
    print(f"\n{title} (block x target):")
    print("        " + " ".join(f"{t:>9s}" for t in ADA_TARGETS))
    for bi in range(cfg.n_layers):
        print(f"block {bi} " + " ".join(f"{r:9d}" for r in rk[bi * len(ADA_TARGETS):(bi + 1) * len(ADA_TARGETS)]))

final_alloc = allocator.mask_pattern()
_rank_table(f"final rank allocation at the end of training (budget {allocator.budget})", final_alloc)
model.load_state_dict(torch.load(SAVE), strict=False)
ranks = allocator.rank_pattern()
_rank_table("ranks in use in the best-epoch weights (non-zero E)", ranks)
used = sum(r * (m.in_features + m.out_features + 1) for r, m in zip(ranks, ada_layers))
print(f"triplets kept {sum(ranks)} / {allocator.init_budget} (target {allocator.target_budget}) | "
      f"adapter values in use {used/1e6:.3f}M")
_best_ep = max(log, key=lambda r: -r["val_loss"] if SELECT_BY == "loss" else
               r.get({"rougeL": "rougeL_f1", "rouge1": "rouge1_f1", "bertscore": "bertscore_f1"}[SELECT_BY]) or 0)
if _best_ep["rank_budget"] > allocator.target_budget:
    print(f"NOTE: the best epoch ({_best_ep['epoch']}) ended before pruning reached the target budget "
          f"(budget then {_best_ep['rank_budget']}).")

t0 = time.time()
val_pred = predict(model, val_ex, GEN_BATCH_SIZE, GEN_KW, MAX_SUMMARY_WORDS, desc="val generate")
val_gen_s = time.time() - t0
print("\n-- held-out val (best epoch)")
res = evaluate_summaries(val_pred, val)
length_buckets(res)
print(f"3-gram repetition rate: pred {repetition_rate(val_pred.values()):.3f} | "
      f"reference {repetition_rate(d['summary'] for d in val.values()):.3f}")
print(f"lead-{LEAD_N} baseline: BERTScore {baseline_res['bertscore_f1']} | R-1 {baseline_res['rouge1_f1']:.4f} | "
      f"R-L {baseline_res['rougeL_f1']:.4f}")
show_examples(val_pred, val)
print(f"training {train_min:.1f} min | val generation {val_gen_s:.0f} s for {len(val_pred)} docs | "
      f"peak training memory {train_mem:.2f} GB")
_cfg_out = {"run": RUN_NAME, "method": "AdaLoRA W + P diag(E) Q", "sep": SEP, "build": {**BUILD, "sep_ids": SEP_IDS},
            "gen": GEN_KW, "ada_targets": list(ADA_TARGETS), "ada_init_rank": ADA_INIT_RANK,
            "ada_target_rank": ADA_TARGET_RANK, "schedule": {"total_steps": TOTAL_STEPS, "tinit_frac": TINIT_FRAC,
            "tfinal_frac": TFINAL_FRAC, "delta_t": DELTA_T, "beta1": BETA1, "beta2": BETA2, "orth_reg": ORTH_REG}}
json.dump({**_cfg_out, "ranks": dict(zip(ada_names, ranks)),
           "final_allocation": dict(zip(ada_names, final_alloc)), "log": log, "coverage_input_vs_doc": cov,
           "train_minutes": train_min, "peak_mem_gb": train_mem,
           "val": {k: res[k] for k in ("bertscore_f1", "rouge1_f1", "rougeL_f1", "pred_words_median",
                                       "ref_words_median", "len_ratio_median")},
           "val_pred": val_pred}, open(f"{OUT_DIR}/val_log.json", "w"), ensure_ascii=False)

# Test summaries with the best weights -> /kaggle/working/<RUN_NAME>/submission.zip
test_pred = predict(model, test_ex, GEN_BATCH_SIZE, GEN_KW, MAX_SUMMARY_WORDS, desc="test generate")
write_prediction(test_pred, out_dir=OUT_DIR)

# [ANCILLARY] everything a KD notebook needs to rebuild this model as the teacher
if DUMP_TRAIN_SUMMARIES:
    train_pred = predict(model, train_ex, GEN_BATCH_SIZE, GEN_KW, MAX_SUMMARY_WORDS, desc="train generate")
    json.dump(train_pred, open(f"{OUT_DIR}/teacher_train_summaries.json", "w"), ensure_ascii=False)
json.dump({**_cfg_out, "ranks": dict(zip(ada_names, ranks)), "weights": "best_trainable.pt",
           "val_ids": sorted(VAL_IDS)}, open(f"{OUT_DIR}/teacher_config.json", "w"), indent=1)
print("files for later runs / KD:", sorted(os.listdir(OUT_DIR)))


SEP = 'eos' -> ids [3] | document budget 1791 tokens
AdaLoRA target names (same in all blocks):
  q_proj     1024 ->  1024  bias=False
  k_proj     1024 ->  1024  bias=False
  v_proj     1024 ->  1024  bias=False
  out_proj   1024 ->  1024  bias=False
  ffn_in     1024 ->  4096  bias=True
  ffn_out    4096 ->  1024  bias=True
AdaLoRALinear = W + P diag(E) Q checked
wrapped 48 linears with AdaLoRA | identity-at-init max |diff| = 6.10e-05 (should be ~0)
trainable 1.770M / total 112.75M (1.57%)


examples:   0%|          | 0/6325 [00:00<?, ?it/s]

6325 docs | document budget 1791 tokens | document tokens kept per doc median 1536 (whole doc 3929) | doc tokens seen 27.4% | summaries cut at DEC_LEN: 6117 (96.7%)


examples:   0%|          | 0/703 [00:00<?, ?it/s]

703 docs | document budget 1791 tokens | document tokens kept per doc median 1536 (whole doc 4010) | doc tokens seen 26.0% | summaries cut at DEC_LEN: 681 (96.9%)


examples:   0%|          | 0/100 [00:00<?, ?it/s]

... (38 outputs left out)

repetition:   0%|          | 0/703 [00:00<?, ?it/s]

3-gram repetition rate: pred 0.000 | reference 0.084
lead-637 baseline: BERTScore 0.8311378359794617 | R-1 0.4939 | R-L 0.2473
--- 5156
PRED: The appellant was not be a suit for the High Court in respect of the State Government to pay it is no jurisdiction under section 3(i) and that there were entitled to this Court, HELD: The respondent 's case on the said order dated January, as an application. Income Tax Act, 1949/708. Only or any other words, the ground that he did not been made by the assessee had become become become his brothers are such person who has been taken into force with effect from the Act which they were also the provisions of the lands of the date of the respondents filed a notice of the first time being a public interested the basis of the parties within
REF : HELD: 1. The fact that the offences, if any, are alleged to have been committed more than six years ago and the appellant was merely a trader at the lowest rung of the hierarchy in the Foreign Exchange Divisio

test generate:   0%|          | 0/25 [00:00<?, ?it/s]

test generate: 100 docs in 1.4 min
wrote /kaggle/working/c1_adalora_summ_spread/prediction.json and /kaggle/working/c1_adalora_summ_spread/submission.zip: 100 summaries | words min 122 median 150 max 150 | empty (written as '.') 0
files for later runs / KD: ['best_trainable.pt', 'last_checkpoint.pt', 'prediction.json', 'submission.zip', 'teacher_config.json', 'val_log.json']
